In [3]:
# ============================================================
# Model 1: SVM + TF-IDF Baseline
# Terminal Table Output
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "/net/pr2/projects/plgrid/plggman01/HateSpeech"

PATH_IDHSD = os.path.join(DATA_DIR, "IDHSD_RIO_unbalanced_713_2017.txt")
PATH_572 = os.path.join(DATA_DIR, "572-hate-speech-dataset.csv")
PATH_RE = os.path.join(DATA_DIR, "re_dataset.csv")
PATH_KAMUSALAY = os.path.join(DATA_DIR, "new_kamusalay.csv")

SEED = 42
LABEL_FRACTIONS = [0.05, 0.10, 0.20]

random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# SAFE READ
# ============================================================

def safe_read_csv(path, sep=",", header="infer"):
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, header=header)
        except Exception as e:
            last_error = e
    raise ValueError(f"Cannot read {path}: {last_error}")

# ============================================================
# LABEL NORMALIZATION
# ============================================================

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"hs", "hate", "hate_speech", "1", "true", "yes"}:
            return 1
        if v in {"non_hs", "non-hs", "non hs", "0", "false", "no"}:
            return 0

    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value)

    return np.nan

# ============================================================
# TEXT CLEANING
# ============================================================

URL_PATTERN = re.compile(r"http\S+|www\S+|https\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
NON_ALNUM_PATTERN = re.compile(r"[^a-zA-Z0-9\s!?]")
MULTISPACE_PATTERN = re.compile(r"\s+")
REPEAT_CHAR_PATTERN = re.compile(r"(.)\1{2,}")

LAUGHTER_VARIANTS = {
    "wkwk": "tertawa",
    "wkwkwk": "tertawa",
    "haha": "tertawa",
    "hahaha": "tertawa",
    "hehe": "tertawa"
}

def reduce_repeated_chars(text):
    return REPEAT_CHAR_PATTERN.sub(r"\1\1", text)

def split_hashtag(match):
    return f" {match.group(1)} "

def normalize_special_tokens(text):
    return " ".join([LAUGHTER_VARIANTS.get(tok, tok) for tok in text.split()])

def basic_clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\\n", " ")
    text = text.replace("\\/", "/")
    text = reduce_repeated_chars(text)
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(split_hashtag, text)
    text = text.lower()
    text = NON_ALNUM_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

def load_kamusalay_dict(path):
    df = safe_read_csv(path)
    c1, c2 = df.columns[:2]
    slang = {}
    for _, row in df.iterrows():
        slang[str(row[c1]).strip().lower()] = str(row[c2]).strip().lower()
    return slang

def slang_normalize(text, slang_dict):
    return " ".join([slang_dict.get(tok, tok) for tok in text.split()])

def preprocess_text(text, slang_dict):
    text = basic_clean_text(text)
    text = normalize_special_tokens(text)
    text = slang_normalize(text, slang_dict)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

# ============================================================
# LOAD DATASETS
# ============================================================

def load_idhsd_dataset(path):
    df = safe_read_csv(path, sep="\t")
    df = df.rename(columns={"Label": "raw_label", "Tweet": "text"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text", "label"]].dropna()

def load_572_dataset(path):
    df = safe_read_csv(path)
    text_col = next((c for c in ["comment_text","tweet","Tweet","text","Text"] if c in df.columns), None)
    label_col = next((c for c in ["class","label","Label","HS"] if c in df.columns), None)
    df = df.rename(columns={text_col:"text", label_col:"raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text", "label"]].dropna()

def load_re_dataset(path):
    df = safe_read_csv(path)
    text_col = next((c for c in ["Tweet","tweet","text","Text"] if c in df.columns), None)
    label_col = "HS" if "HS" in df.columns else "label"
    df = df.rename(columns={text_col:"text", label_col:"raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text", "label"]].dropna()

def merge_datasets():
    slang_dict = load_kamusalay_dict(PATH_KAMUSALAY)

    ds1 = load_idhsd_dataset(PATH_IDHSD)
    ds2 = load_572_dataset(PATH_572)
    ds3 = load_re_dataset(PATH_RE)

    data = pd.concat([ds1, ds2, ds3], ignore_index=True)
    data = data.drop_duplicates(subset=["text"]).reset_index(drop=True)
    data["clean_text"] = data["text"].apply(lambda x: preprocess_text(x, slang_dict))
    data = data[data["clean_text"].str.len() > 2].reset_index(drop=True)

    print("="*70)
    print("FINAL DATASET SIZE :", len(data))
    print("LABEL DISTRIBUTION :", Counter(data["label"]))
    print("="*70)

    return data

# ============================================================
# EVALUATION
# ============================================================

def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

def find_best_threshold(y_true, y_prob):
    best_t = 0.5
    best_f1 = -1
    for t in np.arange(0.20, 0.81, 0.02):
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t

# ============================================================
# BUILD MODEL
# ============================================================

def build_model():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            analyzer="word",
            ngram_range=(1, 2),
            max_features=50000,
            min_df=2,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED,
            solver="liblinear"
        ))
    ])

    return pipe

# ============================================================
# RUN EXPERIMENT
# ============================================================

def run_experiment(train_df, val_df, test_df, frac):
    X_train = train_df["clean_text"].values
    y_train = train_df["label"].astype(int).values

    X_val = val_df["clean_text"].values
    y_val = val_df["label"].astype(int).values

    X_test = test_df["clean_text"].values
    y_test = test_df["label"].astype(int).values

    model = build_model()
    model.fit(X_train, y_train)

    val_prob = model.predict_proba(X_val)[:,1]
    best_t = find_best_threshold(y_val, val_prob)

    test_prob = model.predict_proba(X_test)[:,1]
    test_pred = (test_prob >= best_t).astype(int)

    result = evaluate_predictions(y_test, test_pred, test_prob)
    result["fraction"] = frac
    result["train_size"] = len(train_df)

    return result

# ============================================================
# PRINT TABLE
# ============================================================

def print_results_table(results):
    print("\n" + "="*92)
    print("{:<10} {:<12} {:<10} {:<10} {:<10} {:<10} {:<10}".format(
        "Fraction", "TrainSize", "Accuracy", "MacroF1", "Precision", "Recall", "ROC_AUC"
    ))
    print("="*92)

    for r in results:
        print("{:<10} {:<12} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
            r["fraction"],
            r["train_size"],
            r["accuracy"],
            r["macro_f1"],
            r["precision"],
            r["recall"],
            r["roc_auc"]
        ))

    print("="*92)

# ============================================================
# MAIN
# ============================================================

data = merge_datasets()

train_val_df, test_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED,
    stratify=data["label"]
)

train_df_full, val_df = train_test_split(
    train_val_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_val_df["label"]
)

results = []

for frac in LABEL_FRACTIONS:
    labeled_train_df, _ = train_test_split(
        train_df_full,
        train_size=frac,
        random_state=SEED,
        stratify=train_df_full["label"]
    )

    res = run_experiment(labeled_train_df, val_df, test_df, frac)
    results.append(res)

print_results_table(results)

print("\nLOGISTIC REGRESSION + TFIDF BASELINE FINISHED.")

FINAL DATASET SIZE : 13899
LABEL DISTRIBUTION : Counter({0.0: 7966, 1.0: 5933})

Fraction   TrainSize    Accuracy   MacroF1    Precision  Recall     ROC_AUC   
0.05       500          0.7640     0.7597     0.7590     0.7606     0.8346    
0.1        1000         0.7809     0.7763     0.7761     0.7765     0.8521    
0.2        2001         0.7957     0.7894     0.7928     0.7872     0.8767    

LOGISTIC REGRESSION + TFIDF BASELINE FINISHED.


In [ ]:
import os
os.getcwd()